## Prompt Evaluation (Model Grader Workflow)

This lesson looks at how prompts can be evaluated in a structured process.

The script builds on Lesson 3 where a model grader is included into the workflow.


In [1]:
# Import python's built-in regular expression library
import re
import os
from pathlib import Path
import anthropic
import json

def load_env_file(env_path=".env"):
    env_path = Path(env_path)

    if not env_path.exists():
        raise FileNotFoundError(f"Could not find .env file at: {env_path.resolve()}")

    with env_path.open("r") as file:
        for line in file:
            line = line.strip()

            if not line or line.startswith("#"):
                continue

            if "=" not in line:
                continue

            key, value = line.split("=", 1)
            os.environ[key.strip()] = value.strip().strip('"').strip("'")

load_env_file(".env")

API_KEY = os.environ.get("ANTHROPIC_API_KEY")
MODEL_NAME = os.environ.get("MODEL_NAME")

if not API_KEY:
    raise ValueError("Missing ANTHROPIC_API_KEY in .env")

if not MODEL_NAME:
    raise ValueError("Missing MODEL_NAME in .env")

client = anthropic.Anthropic(api_key=API_KEY)

print("Loaded API key and model name from .env")


Loaded API key and model name from .env


In [2]:
# creating helper fuctions

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": MODEL_NAME,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    response = client.messages.create(**params)
    return response.content[0].text

In [3]:
# Function to generate a new dataset
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)


In [4]:
# Generate the dataset and write it to 'dataset.json'
dataset = generate_dataset()
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [5]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)


In [6]:
# Passes a test case into Claude
def run_prompt(test_case):
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [7]:
# Function to execute a single test case and grade the output
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [8]:
from statistics import mean


def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [9]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 6.666666666666667


In [10]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# Extract AWS Region from S3 Bucket ARN\n\nHere are several approaches to solve this:\n\n## Approach 1: Simple Parsing (Most Reliable)\n```python\ndef extract_region_from_s3_arn(arn):\n    \"\"\"\n    Extract region from S3 bucket ARN.\n    Note: S3 bucket ARNs don't inherently contain region info.\n    Extract region from bucket name if it follows naming convention.\n    \"\"\"\n    # S3 ARN format: arn:aws:s3:::bucket-name\n    bucket_name = arn.split(':::')[-1]\n    \n    # Common AWS regions to check\n    regions = [\n        'us-east-1', 'us-east-2', 'us-west-1', 'us-west-2',\n        'eu-west-1', 'eu-central-1', 'ap-southeast-1', 'ap-southeast-2',\n        'ap-northeast-1', 'ap-south-1', 'ca-central-1', 'sa-east-1'\n    ]\n    \n    for region in regions:\n        if region in bucket_name:\n            return region\n    \n    return None\n\n# Test\narn = 'arn:aws:s3:::my-bucket-us-east-1'\nprint(extract_region_from_s3_arn(arn))  # Output: us-east-1\n```\n\n#